Use requests and BeautifulSoup in Python to scrape the latest 10 headlines, authors, publication dates, and summaries from the technology section of a popular news site like NDTV Gadgets or The Verge, and print the extracted data.

In [1]:
import requests
from bs4 import BeautifulSoup

def scrape_ndtv_tech():
    # Target URL: NDTV Gadgets 360 Technology Section
    url = "https://www.ndtv.com/technology"
    
    # Custom headers to mimic a real browser request and avoid 403 Forbidden blocks
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/122.0.0.0 Safari/537.36"
    }

    print(f"Fetching articles from {url}...")
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching the webpage: {e}")
        return

    # Parse HTML content using BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')

    # Locate article containers on the page
    # Note: Website layouts update frequently; adjust selectors based on live inspection if needed.
    articles = soup.find_all('div', class_='news_Itm')[:10]

    if not articles:
        print("No articles found. The website structure may have changed.")
        return

    print(f"\nSuccessfully retrieved {len(articles)} articles. Processing details...\n" + "="*60)

    for index, article in enumerate(articles, start=1):
        # 1. Headline
        headline_elem = article.find('h2', class_='news_Itm-ttl') or article.find('a', class_='item-title')
        headline = headline_elem.get_text(strip=True) if headline_elem else "N/A"

        # 2. Summary / Description
        summary_elem = article.find('p', class_='news_Itm-des')
        summary = summary_elem.get_text(strip=True) if summary_elem else "N/A"

        # 3. Author & Publication Date
        # NDTV typically groups meta info inside an element like 'news_Itm-by' or similar span/divs
        meta_elem = article.find('span', class_='posted-by') or article.find('div', class_='news_Itm-by')
        meta_text = meta_elem.get_text(strip=True) if meta_elem else "N/A"
        
        # Simple separation if author and date share the same text block
        # (Alternatively, you can parse individual child elements depending on exact HTML)

        print(f"[{index}] Headline : {headline}")
        print(f"    Summary  : {summary}")
        print(f"    Meta     : {meta_text}")
        print("-" * 60)

if __name__ == "__main__":
    scrape_ndtv_tech()

Fetching articles from https://www.ndtv.com/technology...
Error fetching the webpage: 403 Client Error: Forbidden for url: https://www.ndtv.com/technology


Store the scraped news data (title, author, publication date, summary) into a pandas DataFrame and save it as a CSV file named tech_news.csv.

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_and_save_tech_news():
    url = "https://www.ndtv.com/technology"
    
    # Custom headers to avoid 403 Forbidden blocks
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/122.0.0.0 Safari/537.36"
    }

    print(f"Fetching articles from {url}...")
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching the webpage: {e}")
        return

    # Parse HTML content
    soup = BeautifulSoup(response.text, 'html.parser')
    articles = soup.find_all('div', class_='news_Itm')[:10]

    if not articles:
        print("No articles found. The website structure may have changed.")
        return

    # List to store dictionaries of scraped data
    news_data = []

    for article in articles:
        # 1. Title / Headline
        headline_elem = article.find('h2', class_='news_Itm-ttl') or article.find('a', class_='item-title')
        title = headline_elem.get_text(strip=True) if headline_elem else "N/A"

        # 2. Summary
        summary_elem = article.find('p', class_='news_Itm-des')
        summary = summary_elem.get_text(strip=True) if summary_elem else "N/A"

        # 3. Author & Publication Date (extracted from meta block)
        meta_elem = article.find('span', class_='posted-by') or article.find('div', class_='news_Itm-by')
        meta_text = meta_elem.get_text(strip=True) if meta_elem else "N/A"
        
        # Simple split if author and date share the string (e.g., "By Author | Date")
        parts = [p.strip() for p in meta_text.split('|')]
        author = parts[0] if len(parts) > 0 else "N/A"
        pub_date = parts[1] if len(parts) > 1 else "N/A"

        # Append structured dictionary
        news_data.append({
            "Title": title,
            "Author": author,
            "Publication Date": pub_date,
            "Summary": summary
        })

    # 4. Convert list of dictionaries into a pandas DataFrame
    df = pd.DataFrame(news_data)

    # 5. Export DataFrame to CSV
    output_filename = "tech_news.csv"
    df.to_csv(output_filename, index=False)

    print(f"\nSuccess! Scraped {len(df)} articles and saved them to '{output_filename}'.")
    
    # Display a preview of the DataFrame
    print("\nData Preview:")
    print(df[['Title', 'Author']].head())

if __name__ == "__main__":
    scrape_and_save_tech_news()

Fetching articles from https://www.ndtv.com/technology...
Error fetching the webpage: 403 Client Error: Forbidden for url: https://www.ndtv.com/technology


Load your tech_news.csv file into pandas and perform basic EDA: display the number of articles, count how many articles each author wrote, and show the earliest and latest publication dates.

In [3]:
import pandas as pd

def perform_eda():
    filename = "tech_news.csv"
    
    try:
        # 1. Load the dataset into a pandas DataFrame
        print(f"Loading '{filename}'...")
        df = pd.read_csv(filename)
        
        # 2. Display the total number of articles
        total_articles = len(df)
        print(f"\n--- Total Articles ---")
        print(f"Total number of articles: {total_articles}")
        
        # 3. Count how many articles each author wrote
        print(f"\n--- Articles Per Author ---")
        if "Author" in df.columns:
            author_counts = df["Author"].value_counts()
            print(author_counts)
        else:
            print("Column 'Author' not found in the dataset.")
            
        # 4. Show the earliest and latest publication dates
        print(f"\n--- Publication Date Range ---")
        date_col = "Publication Date"
        if date_col in df.columns:
            # Convert to datetime, coercing unparseable strings to NaT (Not a Time)
            df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
            
            earliest_date = df[date_col].min()
            latest_date = df[date_col].max()
            
            print(f"Earliest Publication Date: {earliest_date}")
            print(f"Latest Publication Date: {latest_date}")
        else:
            print(f"Column '{date_col}' not found in the dataset.")

    except FileNotFoundError:
        print(f"Error: The file '{filename}' was not found. Please ensure you have run the scraper script first.")
    except Exception as e:
        print(f"An unexpected error occurred during EDA: {e}")

if __name__ == "__main__":
    perform_eda()

Loading 'tech_news.csv'...
Error: The file 'tech_news.csv' was not found. Please ensure you have run the scraper script first.


Add a feature to your scraper so it skips any article where the summary is missing or empty before saving to CSV.<br><br><em><strong>Hint:</strong> Use an if condition to check for empty summaries during extraction.</em>

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_and_save_filtered_news():
    url = "https://www.ndtv.com/technology"
    
    # Custom headers to avoid 403 Forbidden blocks
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/122.0.0.0 Safari/537.36"
    }

    print(f"Fetching articles from {url}...")
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching the webpage: {e}")
        return

    # Parse HTML content
    soup = BeautifulSoup(response.text, 'html.parser')
    articles = soup.find_all('div', class_='news_Itm')

    if not articles:
        print("No articles found. The website structure may have changed.")
        return

    news_data = []
    skipped_count = 0

    for article in articles:
        # 1. Title / Headline
        headline_elem = article.find('h2', class_='news_Itm-ttl') or article.find('a', class_='item-title')
        title = headline_elem.get_text(strip=True) if headline_elem else "N/A"

        # 2. Summary
        summary_elem = article.find('p', class_='news_Itm-des')
        summary = summary_elem.get_text(strip=True) if summary_elem else ""

        # 3. Validation Check: Skip if summary is missing, empty, or whitespace-only
        if not summary or summary.isspace():
            skipped_count += 1
            continue  # Skip this iteration and move to the next article

        # 4. Author & Publication Date
        meta_elem = article.find('span', class_='posted-by') or article.find('div', class_='news_Itm-by')
        meta_text = meta_elem.get_text(strip=True) if meta_elem else "N/A"
        
        parts = [p.strip() for p in meta_text.split('|')]
        author = parts[0] if len(parts) > 0 else "N/A"
        pub_date = parts[1] if len(parts) > 1 else "N/A"

        # Append structured dictionary only if valid
        news_data.append({
            "Title": title,
            "Author": author,
            "Publication Date": pub_date,
            "Summary": summary
        })

        # Stop once we have successfully collected 10 valid articles
        if len(news_data) >= 10:
            break

    # 5. Convert list of dictionaries into a pandas DataFrame and save
    if news_data:
        df = pd.DataFrame(news_data)
        output_filename = "tech_news.csv"
        df.to_csv(output_filename, index=False)
        
        print(f"\nSuccess! Saved {len(df)} valid articles to '{output_filename}' (Skipped {skipped_count} articles with missing summaries).")
    else:
        print("No articles with valid summaries were found.")

if __name__ == "__main__":
    scrape_and_save_filtered_news()

Fetching articles from https://www.ndtv.com/technology...
Error fetching the webpage: 403 Client Error: Forbidden for url: https://www.ndtv.com/technology


Use ChatGPT to help you write a function that cleans the article summaries by removing extra whitespace and any HTML tags, then apply this function to your DataFrame before saving the CSV.<br><br><em><strong>Hint:</strong> Paste your current summary extraction code into ChatGPT and ask for suggestions on cleaning text data.</em>

In [5]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Remove HTML tags
    no_html = re.sub(r'<[^>]+>', '', text)
    # Replace multiple spaces, newlines, and tabs with a single space and strip edges
    cleaned = re.sub(r'\s+', ' ', no_html).strip()
    return cleaned